In [ ]:
import os
from dotenv import load_dotenv
import pandas as pd
import numpy as np
import statsmodels.api as sm
from tiingo import TiingoClient
from scipy.optimize import minimize

load_dotenv()  # Load environment variables from .env file

In [ ]:
# --- 1. Setup & Configuration ---
config = {
    'session': True,
    'api_key': os.getenv("TIINGO_API_KEY") 
}
client = TiingoClient(config)

# The "Twist" Universe + SPY for Benchmark
TICKERS = ['SPY', 'PLTR', 'NVDA', 'INTC', 'GLD', 'EEM']
ASSETS = ['PLTR', 'NVDA', 'INTC', 'GLD', 'EEM'] # Excludes SPY for portfolio

START_DATE = '2023-01-01'
END_DATE = '2026-01-01'

In [ ]:
from fredapi import Fred

# --- 1.5 FRED Risk-Free Rate Retrieval ---
if 'fred_rf' not in globals():
    print("Fetching 3-Month Treasury data via FRED API...")
    fred = Fred(api_key=os.getenv("FRED_API_KEY"))
    
    # Pull the DGS3MO series
    fred_series = fred.get_series('DGS3MO', observation_start=START_DATE, observation_end=END_DATE)
    
    # Format to DataFrame and convert to daily decimal
    fred_rf = pd.DataFrame(fred_series, columns=['Daily_Rf'])
    fred_rf['Daily_Rf'] = fred_rf['Daily_Rf'] / 100 / 252
    print("FRED data loaded and formatted.")
else:
    print("Using cached FRED data from memory.")

In [ ]:
# --- 2. Data Retrieval & Processing ---
print("Fetching data from Tiingo...")

if 'prices' not in globals():
    print("Fetching equity data from Tiingo...")
    prices = pd.DataFrame()
    
    for ticker in TICKERS:
        # Dropped metric_name to prevent the Series/DateParseError issue
        data = client.get_dataframe(ticker, startDate=START_DATE, endDate=END_DATE)
        prices[ticker] = data['adjClose']
        print(f"Pulled {ticker}...")
        
    print("Tiingo data fully loaded.")
else:
    print("Using cached Tiingo prices from memory.")


In [ ]:
# Calculate daily simple returns (standard for portfolio math)
returns = prices.pct_change().dropna()

returns.index = returns.index.tz_localize(None)


returns = returns.join(fred_rf, how='left').ffill().dropna()

print("Returns calculated and merged with Risk-Free rate.")


In [13]:
# --- 3. Summary Statistics & CAPM Regressions ---
print("\n--- Summary Statistics (Daily) ---")
summary_stats = pd.DataFrame(index=ASSETS)
summary_stats['Avg Daily Return'] = returns[ASSETS].mean()
summary_stats['Daily Std Dev'] = returns[ASSETS].std()

returns.rename(columns={'Daily_Rf': 'DAILY_RF'}, inplace=True)  # Rename for easier access

# Calculate Beta against SPY
betas = []

# FIX: Add .rename('SPY') so statsmodels keeps the variable name
market_excess = (returns['SPY'] - returns['DAILY_RF']).rename('SPY')
X = sm.add_constant(market_excess)

X = sm.add_constant(market_excess) # Market excess return
for asset in ASSETS:
    y = returns[asset] - returns['DAILY_RF'] # Asset excess return
    model = sm.OLS(y, X).fit()
    betas.append(model.params['SPY'])

summary_stats['Beta (vs SPY)'] = betas
print(summary_stats.round(6))




--- Summary Statistics (Daily) ---
      Avg Daily Return  Daily Std Dev  Beta (vs SPY)
PLTR          0.005292       0.042140       2.277246
NVDA          0.003917       0.031637       2.085854
INTC          0.001023       0.033001       1.646792
GLD           0.001172       0.010288       0.086401
EEM           0.000629       0.010222       0.719968


In [15]:
# --- 4. Covariance & Correlation Matrices ---
print("\n--- Correlation Matrix ---")
corr_matrix = returns[ASSETS].corr()
print(corr_matrix.round(4))

print("\n--- V-Cov Matrix ---")
cov_matrix = returns[ASSETS].cov()
print(cov_matrix.round(4))



--- Correlation Matrix ---
        PLTR    NVDA    INTC     GLD     EEM
PLTR  1.0000  0.4048  0.2681  0.0462  0.3817
NVDA  0.4048  1.0000  0.2921  0.0173  0.4545
INTC  0.2681  0.2921  1.0000  0.0757  0.3908
GLD   0.0462  0.0173  0.0757  1.0000  0.3133
EEM   0.3817  0.4545  0.3908  0.3133  1.0000

--- V-Cov Matrix ---
        PLTR    NVDA    INTC     GLD     EEM
PLTR  0.0018  0.0005  0.0004  0.0000  0.0002
NVDA  0.0005  0.0010  0.0003  0.0000  0.0001
INTC  0.0004  0.0003  0.0011  0.0000  0.0001
GLD   0.0000  0.0000  0.0000  0.0001  0.0000
EEM   0.0002  0.0001  0.0001  0.0000  0.0001


In [18]:

# --- 5. Portfolio Optimization (Unconstrained / Allowing Shorts) ---
# Objective Function: Negative Sharpe Ratio (to minimize)
def neg_sharpe(weights, mean_returns, cov_matrix, avg_daily_rf):
    port_return = np.sum(mean_returns * weights)
    port_std = np.sqrt(np.dot(weights.T, np.dot(cov_matrix, weights)))
    
    # Annualize for Sharpe calculation
    ann_return = port_return * 252
    ann_std = port_std * np.sqrt(252)
    ann_rf = avg_daily_rf * 252  # <-- Annualizing the dynamic Rf here
    
    sharpe = (ann_return - ann_rf) / ann_std
    return -sharpe

# Constraints: Weights must sum to 1
constraints = ({'type': 'eq', 'fun': lambda w: np.sum(w) - 1})

# Bounds: Allowing shorting. Limiting to -100% to +200% to prevent absurd solver extremes
bounds = tuple((-1.0, 2.0) for _ in range(len(ASSETS)))
init_guess = len(ASSETS) * [1. / len(ASSETS),]

# Run Optimization
opt_results = minimize(neg_sharpe, init_guess, 
                       args=(returns[ASSETS].mean(), cov_matrix, returns['DAILY_RF'].mean()),
                       method='SLSQP', bounds=bounds, constraints=constraints)

optimal_weights = opt_results.x


In [23]:
# --- 6. Final Exam Outputs ---
print("\n--- Unconstrained Tangency Portfolio ---")
weights_df = pd.DataFrame(data=optimal_weights, index=ASSETS, columns=['Optimal Weight'])
print(weights_df.round(4))

# Calculate final portfolio metrics
opt_daily_ret = np.sum(returns[ASSETS].mean() * optimal_weights)
opt_daily_std = np.sqrt(np.dot(optimal_weights.T, np.dot(cov_matrix, optimal_weights)))

opt_ann_ret = opt_daily_ret * 252
opt_ann_std = opt_daily_std * np.sqrt(252)

# FIX: Re-calculate the annualized risk-free rate in this scope
ann_rf = returns['DAILY_RF'].mean() * 252

# Calculate the final Sharpe ratio
opt_sharpe = (opt_ann_ret - ann_rf) / opt_ann_std

print(f"\nAnnualized Expected Return: {opt_ann_ret:.2%}")
print(f"Annualized Standard Deviation: {opt_ann_std:.2%}")
print(f"Maximized Sharpe Ratio: {opt_sharpe:.4f}")


--- Unconstrained Tangency Portfolio ---
      Optimal Weight
PLTR          0.2729
NVDA          0.3994
INTC         -0.0456
GLD           1.2331
EEM          -0.8599

Annualized Expected Return: 97.45%
Annualized Standard Deviation: 32.07%
Maximized Sharpe Ratio: 2.8860
